# Sustentabilidade e Desenvolvimento nos Municípios Portugueses
## Análise de Clusters com dados PORDATA

**Autor:** Marcus Corrêa Lopes Guedes  
**Curso:** Análise de Dados com Python — IPVC  
**Versão:** Portfolio Edition

### Pergunta
Os municípios portugueses apresentam perfis homogéneos de desenvolvimento e sustentabilidade ou é possível identificar grupos distintos com características próprias?

Esta versão revisa o notebook acadêmico original, elimina dados simulados e utiliza apenas informações derivadas dos 25 arquivos PORDATA fornecidos.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import f_oneway

ROOT = Path("..")
DATA = ROOT / "data" / "processed" / "pordata_municipios_2024.csv"
df = pd.read_csv(DATA)

print(f"{df.shape[0]} municípios × {df.shape[1]} colunas")
display(df.head())


## Seleção de variáveis

O modelo final evita redundância direta entre proporções etárias. Como `Jovens + IdadeAtiva + Idosos ≈ 100%`, manter as três variáveis daria peso excessivo à mesma dimensão demográfica.

Também excluímos a população absoluta para reduzir o efeito do porte municipal. A análise utiliza:

- Densidade populacional
- Percentagem de idosos
- Desemprego
- Resíduos seletivos per capita
- Consumo de energia por habitante
- Variação populacional 2011–2024
- Alunos do ensino superior por 1.000 habitantes


In [ ]:
FEATURES = [
    "Densidade",
    "Idosos",
    "Desemprego",
    "Residuos",
    "Energia",
    "VariacaoPop",
    "EnsinoSuperior_por1000",
]

X = df[FEATURES].copy()
display(X.describe().T.round(2))


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scores = {}
inertias = {}

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=100)
    labels = model.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
    inertias[k] = model.inertia_
    print(f"k={k}: silhueta={scores[k]:.3f} | inércia={inertias[k]:.2f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(scores.keys()), list(scores.values()), marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Coeficiente de silhueta")
plt.title("Seleção do número de clusters")
plt.grid(alpha=0.3)
plt.show()

k_final = max(scores, key=scores.get)
print("k selecionado:", k_final)


In [ ]:
model = KMeans(n_clusters=k_final, random_state=42, n_init=100)
df["Cluster"] = model.fit_predict(X_scaled)

for cluster in sorted(df["Cluster"].unique()):
    nomes = df.loc[df["Cluster"] == cluster, "Municipio"].sort_values().tolist()
    print(f"Cluster {cluster} ({len(nomes)}):")
    print(", ".join(nomes))
    print()


In [ ]:
perfil = df.groupby("Cluster")[FEATURES].mean().round(2)
display(perfil)


## Validação estatística

A ANOVA é usada de forma exploratória para verificar quais variáveis apresentam diferenças de média entre os grupos. Como a amostra é pequena, os resultados devem ser interpretados com cautela.

O teste Qui-Quadrado entre região e cluster não é usado como evidência inferencial nesta versão: com 25 municípios e várias categorias regionais, as frequências esperadas ficam abaixo das condições usuais do teste.


In [ ]:
anova = []
for feature in FEATURES:
    grupos = [
        df.loc[df["Cluster"] == c, feature].values
        for c in sorted(df["Cluster"].unique())
    ]
    f_stat, p_value = f_oneway(*grupos)
    anova.append((feature, f_stat, p_value))

anova_df = pd.DataFrame(anova, columns=["Variavel", "F", "p_value"])
anova_df["Significativa_5pct"] = anova_df["p_value"] < 0.05
display(anova_df.round(4))


In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

plot_df = df[["Municipio", "Cluster"]].copy()
plot_df["PC1"] = coords[:, 0]
plot_df["PC2"] = coords[:, 1]

plt.figure(figsize=(10, 7))
for cluster in sorted(plot_df["Cluster"].unique()):
    parte = plot_df[plot_df["Cluster"] == cluster]
    plt.scatter(parte["PC1"], parte["PC2"], label=f"Cluster {cluster}", s=60)

for _, row in plot_df.iterrows():
    plt.annotate(row["Municipio"], (row["PC1"], row["PC2"]), fontsize=8, xytext=(3, 3), textcoords="offset points")

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("Visualização dos clusters por PCA")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print(f"Variância explicada por PC1 + PC2: {pca.explained_variance_ratio_.sum():.1%}")


## Conclusão

Com a seleção de variáveis revisada, a melhor solução entre `k=2` e `k=6` é **k=3**, com coeficiente de silhueta de aproximadamente **0.291**.

O resultado indica que existem padrões diferenciados entre os municípios analisados, mas a separação não é forte o suficiente para tratar os clusters como categorias rígidas. O uso mais adequado é **exploratório**, servindo para identificar perfis relativos e apoiar discussões sobre desenvolvimento territorial.

A versão revisada prioriza reprodutibilidade e transparência metodológica, evitando conclusões estatísticas não sustentadas pelos dados.
